In [127]:
# Helper Functions

import os
import math
import copy
import open3d as o3d
import numpy as np
from scipy.spatial.transform import Rotation as R



def get_rotation_matrix_z(deg):
    """Creates a 3x3 rotation matrix for the Z-axis."""
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0], 
                     [s,  c, 0], 
                     [0,  0, 1]])


def merge_multiview_scan(data_dir, initial_pos, initial_angle, box_size):
    """Merges multiple view PCDs and removes points below the detected plane."""
    pcd_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.pcd') and "view0" in f])
    merged_pcd = o3d.geometry.PointCloud()
    
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos), 
        R=get_rotation_matrix_z(-initial_angle), 
        extent=np.array(box_size)
    )

    for file_name in pcd_files:
        pcd = o3d.io.read_point_cloud(os.path.join(data_dir, file_name))
   
        # 1. Detect the plane
        plane_model, inliers = pcd.segment_plane(distance_threshold=3.0, ransac_n=3, num_iterations=2000)
        [a, b, c, d] = plane_model

        # 2. Extract all points as a numpy array
        pts = np.asarray(pcd.points)

        # 3. Calculate distance to plane for every point: ax + by + cz + d
        # Points on the plane result in 0. Points above are positive, below are negative.
        distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
        
        # 4. Create an index of points that are ABOVE the plane
        # A small offset (e.g., 0.5mm) to avoid keeping table noise
        above_plane_indices = np.where(distances > 0.5)[0]
        pcd = pcd.select_by_index(above_plane_indices)

        # 5. Crop to the OBB and merge
        merged_pcd += pcd.crop(obb)

    return merged_pcd



def process_point_cloud(pcd, initial_pos, initial_angle, box_size):
    """Processes a single PCD file: removes ground plane and crops to OBB."""
    distance_threshold = 3.0 # mm, for RANSAC plane segmentation
    ransac_n = 3
    num_iterations = 2000
    plane_offset = 0.5 # mm, to ensure we keep points just above the plane

    # 1. Load the specific file    
    if pcd.is_empty():
        print(f"Warning: Point cloud is empty or not found.")
        return pcd

    # 2. Define the Oriented Bounding Box (OBB)
    # Using your rotation logic to align with the YOLO/Initial guess
    rot_matrix = get_rotation_matrix_z(-initial_angle) 
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos), 
        R=rot_matrix, 
        extent=np.array(box_size)
    )

    # 3. Detect and remove the ground plane (RANSAC)
    plane_model, inliers = pcd.segment_plane(distance_threshold=distance_threshold, 
                                             ransac_n=ransac_n, 
                                             num_iterations=num_iterations)
    [a, b, c, d] = plane_model

    # 4. Filter points above the plane
    pts = np.asarray(pcd.points)
    # ax + by + cz + d > offset
    distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
    above_plane_indices = np.where(distances > plane_offset)[0] 
    pcd_filtered = pcd.select_by_index(above_plane_indices)
    # 5. Crop to the region of interest
    # o3d.visualization.draw_geometries([final_pcd, obb], window_name="Downsampled Clouds with Normals")
    final_pcd = pcd_filtered.crop(obb)

    return final_pcd


def load_viewpoint_data(actual_dir, sim_dir):
    """
    Loads all actual and simulated point clouds automatically.

    Args:
        actual_dir (str): Path to 'pcd_data' folder
        sim_dir (str): Path to 'viewpoints_candidate' folder

    Returns:
        target_clouds (list): list of actual point clouds
        source_clouds (list): list of simulated point clouds
    """

    target_clouds = []
    source_clouds = []

    # find all actual viewpoint files
    actual_files = sorted([f for f in os.listdir(actual_dir) if f.endswith(".pcd")])

    for file in actual_files:

        # extract index from filename (view00.pcd -> 0)
        index = int(file.replace("view", "").replace(".pcd", ""))

        sim_name = f"viewpoint_simulated_{index}.pcd"

        actual_path = os.path.join(actual_dir, file)
        sim_path = os.path.join(sim_dir, sim_name)

        try:
            actual_pcd = o3d.io.read_point_cloud(actual_path)
            sim_pcd = o3d.io.read_point_cloud(sim_path)

            target_clouds.append(actual_pcd)
            source_clouds.append(sim_pcd)

        except Exception as e:
            print(f"Error loading index {index}: {e}")

    return target_clouds, source_clouds


def pose_to_matrix(pose):
    """
    Converts [x, y, z, roll, pitch, yaw] in degrees to a 4x4 transformation matrix.
    If zyz=True, assumes the input is in ZYZ Euler Angles, otherwise XYZ Euler Angles.
    
    Args:
    - pose (list or array): [x, y, z, roll, pitch, yaw] in degrees.
    - zyz (bool): If True, interprets the last three values as ZYZ Euler angles. 
                  If False, uses XYZ Euler angles (default).
    
    Returns:
    - T (ndarray): The 4x4 homogeneous transformation matrix.
    """
    x, y, z = pose[:3]   # Translation vector
    roll, pitch, yaw = pose[3:]  # Rotation angles in degrees
    
    # Convert XYZ Euler Angles to a 3x3 rotation matrix
    rot_matrix = R.from_euler('xyz', [roll, pitch, yaw], degrees=True).as_matrix()

    # Build the 4x4 Homogeneous Transformation Matrix
    T = np.eye(4)  # Start with the identity matrix (4x4)
    T[:3, :3] = rot_matrix  # Set the upper-left 3x3 part to the rotation matrix
    T[:3, 3] = [x, y, z]    # Set the upper-right 3x1 part to the translation vector
    return T


def preprocess_normal(pcd, num_points=False, invert_normals=False):
    """Downsamples, estimates normals, and computes FPFH features."""
    if num_points:
        pcd_down = pcd.farthest_point_down_sample(num_points)
    else:
        pcd_down = pcd
    avg_dist = np.mean(pcd_down.compute_nearest_neighbor_distance())
    
    # Estimate Normals
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 2, max_nn=30))
    
    # Orientation fix: Ensure normals point 'up' (positive Z)
    normals = np.asarray(pcd_down.normals)
    
    if invert_normals:
        for i in range(len(normals)):
            if normals[i][2] < 0:
                normals[i] *= -1

    # Compute FPFH (Feature descriptors for global matching)
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down, o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 5, max_nn=100))
    
    return pcd_down, fpfh


def run_global_registration(source, target, source_fpfh, target_fpfh, voxel_size):
    """
    Performs RANSAC-based global registration to find a rough alignment.
    """
    distance_threshold = voxel_size * 1.5
    
    # RANSAC based on feature matching
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source, target, source_fpfh, target_fpfh, 
        mutual_filter=True,
        max_correspondence_distance=distance_threshold,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=3, 
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
        ], 
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
    )
    return result


def run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh):
    """
    Performs iterative RANSAC-based global registration to find the best rough alignment.
    Matches the logic of testing multiple thresholds to find the highest fitness and lowest RMSE.
    """
    max_attempts = 1
    best_fitness = -0.1
    best_inlier_rmse = 100.0
    best_result = None
    best_threshold = None 

    # Thresholds to test, from coarse (10.0) to fine (3.0)
    thresholds = [10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]
    # thresholds = [7]

    for attempt in range(max_attempts):
        for thr in thresholds:            
            result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
                source_down, target_down, source_fpfh, target_fpfh, 
                mutual_filter=True, # Improved matching
                max_correspondence_distance=thr,
                estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
                ransac_n=3, 
                checkers=[
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.85),
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(thr)
                ], 
                criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(10000, 0.99)
            )

            # Update best result if fitness is good and RMSE is lower
            if result.fitness > 0.85 and result.inlier_rmse < best_inlier_rmse:
                best_fitness = result.fitness
                best_inlier_rmse = result.inlier_rmse
                best_result = result
                best_threshold = thr
            
            # Early exit if we find a very high-quality match
            if best_fitness > 0.95 and best_inlier_rmse < 2.35: 
                print(f"Excellent Global Fit Found at Threshold {best_threshold}")
                return best_result, best_threshold
            
    if best_result is None:
        print("Warning: RANSAC could not find a fit above 0.85 fitness.")
        # Fallback to the last result generated if nothing met the 0.85 criteria
        return result, thr

    print(f"RANSAC Finished. Best Threshold: {best_threshold} | Fitness: {best_fitness:.4f}")
    return best_result, best_threshold


# One time local refinement using ICP
def run_local_refinement(source, target, initial_transformation, voxel_size):
    """
    Performs ICP registration to refine the alignment found by RANSAC.
    """
    # We use a smaller threshold for ICP to ensure high precision
    distance_threshold = voxel_size * 0.4
    
    result = o3d.pipelines.registration.registration_icp(
        source, target, distance_threshold, initial_transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
    )
    return result


# Progressive Local Refinement using ICP
def run_local_refinement_adaptive(source, target, initial_trans, best_ransac_thr):
    """
    Refines alignment using an iterative ICP loop. 
    Starts with a coarse threshold and progressively tightens for precision.
    """
    # Define a range of multipliers to tighten the search radius
    # multipliers = [1.0, 0.8, 0.5, 0.25]
    multipliers = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
    # multipliers = [0.6]
    thresholds = [best_ransac_thr * m for m in multipliers]
    
    best_result = None
    best_inlier_rmse = float('inf') # Start with the highest possible error
    
    print(f"{'Threshold':<12} | {'Fitness':<12} | {'RMSE':<12}")
    print("-" * 45)

    for thr in thresholds:
        # Execute Point-to-Point ICP
        reg_icp = o3d.pipelines.registration.registration_icp(
            source, target, thr, initial_trans,
            o3d.pipelines.registration.TransformationEstimationPointToPoint(),
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=1000)
        )

        # Logging the current iteration results
        print(f"{thr:<12.2f} | {reg_icp.fitness:<12.4f} | {reg_icp.inlier_rmse:<12.4f}")

        # Selection Logic: Prioritize lowest RMSE (highest precision) 
        # as long as the fitness is acceptable (e.g., > 85%)
        if reg_icp.fitness > 0.85 and reg_icp.inlier_rmse < best_inlier_rmse:
            best_inlier_rmse = reg_icp.inlier_rmse
            best_result = reg_icp

    # Fallback: if no run met the 85% fitness, return the last result
    return best_result if best_result is not None else reg_icp

In [ ]:
# Create a initial alignment from every viewpoint

# --- 1. Configuration ---
DATA_DIR = "pcd_data"
SOURCE_DIR = "viewpoints_candidate"
SOURCE_PATH = "workpiece/first_object_10000.pcd" #  CAD model

tf_obj = np.load(r'pcd_data/initial_obj_pose.npy')
T_base2ob_yolo = np.load(r'pcd_data/T_base2ob_yolo.npy')

YOLO_POS = tf_obj[:3]                                   # Initial guess from YOLO
YOLO_ANGLE = tf_obj[4]                                  # Initial angle from YOLO
CROP_BOX = [95, 75, 70]                               # Region of interest size


# --- 2. Data Preparation ---
print("Step 1: Merging and cleaning scans...")
source_cloud = o3d.io.read_point_cloud(SOURCE_PATH)
target_cloud = merge_multiview_scan(DATA_DIR, YOLO_POS, YOLO_ANGLE, CROP_BOX)

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0, origin=[0, 0, 0])
o3d.visualization.draw_geometries([target_cloud, source_cloud, world_frame], window_name="Merged Target Cloud")


# 3. Preprocess Normal
source_cloud_transformed = copy.deepcopy(source_cloud).transform(T_base2ob_yolo)
source_down, source_fpfh = preprocess_normal(source_cloud_transformed)
target_down, target_fpfh = preprocess_normal(target_cloud, invert_normals=True)

source_down.paint_uniform_color([1, 0, 0])
target_down.paint_uniform_color([0, 0.651, 0.929])
o3d.visualization.draw_geometries([target_down, source_down], window_name="Processed Target Cloud")



IsADirectoryError: [Errno 21] Is a directory: 'pcd_data'

In [79]:
# --- 3. Global Alignment (RANSAC) ---
print("Step 2: Running RANSAC Global Registration...")
ransac_res, best_thr = run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh)
print(ransac_res)

# Visualize RANSAC result
source_temp = copy.deepcopy(source_down)
source_temp.transform(ransac_res.transformation)
source_temp.paint_uniform_color([1, 0, 0])
target_down.paint_uniform_color([0, 0.651, 0.929])
o3d.visualization.draw_geometries([source_temp, target_down], window_name="RANSAC Result")

Step 2: Running RANSAC Global Registration...
[Open3D WARNING] Too few correspondences (173) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (173) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (173) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (173) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (173) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (173) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (173) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (173) after mutual filter, fall back to original correspondences.
RANSAC Finished. Best Threshold: 9.0 | Fitness: 0.8731
RegistrationResult with fitness=8.731000e-0

In [107]:
# --- 4. Local Alignment (ICP) ---
print("Step 3: Running ICP Local Refinement...")
icp_res = run_local_refinement_adaptive(source_down, target_down, ransac_res.transformation, best_thr)
print(icp_res)

# --- 5. Extract Final Results ---
fine_correction_transformation = icp_res.transformation # just the final ICP result
merge_full_transformation = fine_correction_transformation @ T_base2ob_yolo # Combine the initial YOLO-based transformation with the final ICP correction

initial_POS = merge_full_transformation[:3, 3] 
initial_ORI = merge_full_transformation[:3, :3] 
print("="*30)
print(f"Final Position: {initial_POS}")
print(f"Final Orientation Matrix:\n{initial_ORI}")
print(f"Fitness: {icp_res.fitness:.4f}")
print(f"RMSE: {icp_res.inlier_rmse:.4f}")


# --- 6. Final Visualization ---
source_final = copy.deepcopy(source_cloud).transform(merge_full_transformation)
source_final.paint_uniform_color([1, 0, 0])         # Red: CAD Model
target_cloud.paint_uniform_color([0, 0.65, 0.93])   # Blue: Scanned Data
o3d.visualization.draw_geometries([source_final, target_cloud])

# Save the final transformation for use in the evaluation step
np.save(r'evaluation_result/merge_full_transformation.npy', merge_full_transformation)

Step 3: Running ICP Local Refinement...
Threshold    | Fitness      | RMSE        
---------------------------------------------
9.00         | 1.0000       | 1.6898      
8.10         | 1.0000       | 1.6894      
7.20         | 1.0000       | 1.6898      
6.30         | 0.9994       | 1.6822      
5.40         | 0.9966       | 1.6558      
4.50         | 0.9897       | 1.6103      
3.60         | 0.9607       | 1.4810      
2.70         | 0.8450       | 1.0961      
1.80         | 0.7131       | 0.9006      
0.90         | 0.3007       | 0.6193      
RegistrationResult with fitness=9.607000e-01, inlier_rmse=1.480967e+00, and correspondence_set size of 9607
Access transformation to get result.
Final Position: [593.60862274 117.60379512  -1.43447754]
Final Orientation Matrix:
[[ 0.00765777 -0.99969892 -0.02331138]
 [ 0.99979343  0.00809324 -0.0186437 ]
 [ 0.01882675 -0.0231638   0.9995544 ]]
Fitness: 0.9607
RMSE: 1.4810


In [ ]:
# empty #

In [ ]:
# empty #

In [ ]:
# Viewpoint Evaluation

# --- 1. Configuration ---
DATA_DIR = "pcd_data"
SOURCE_DIR = "viewpoints_candidate"
SOURCE_PATH = "workpiece/first_object_10000.pcd" #  CAD model

tf_obj = np.load(r'pcd_data/initial_obj_pose.npy')
merge_full_transformation = np.load(r'evaluation_result/merge_full_transformation.npy')
YOLO_POS = tf_obj[:3]                                   # Initial guess from YOLO
YOLO_ANGLE = tf_obj[4]                                  # Initial angle from YOLO
CROP_BOX = [95, 75, 70]                               # Region of interest size

target_clouds, source_clouds = load_viewpoint_data(DATA_DIR, SOURCE_DIR)

# # For Testing one viewpoint
# target_cloud = process_point_cloud(target_clouds[7], YOLO_POS, YOLO_ANGLE, CROP_BOX)
# o3d.visualization.draw_geometries([target_cloud], window_name="Processed Target Cloud")

# Aligment Point Clouds
for i in range(len(target_clouds)):
    target_cloud = process_point_cloud(target_clouds[i], YOLO_POS, YOLO_ANGLE, CROP_BOX)
    source_cloud_transformed = copy.deepcopy(source_clouds[i]).transform(merge_full_transformation)

    source_cloud_transformed.paint_uniform_color([1, 0, 0])
    target_cloud.paint_uniform_color([0, 0.651, 0.929])
    # o3d.visualization.draw_geometries([target_cloud, source_cloud_transformed], window_name="Processed Target Cloud")


    # Chamfer distance computation
    dist_sim_to_real = np.asarray(source_cloud_transformed.compute_point_cloud_distance(target_cloud))
    dist_real_to_sim = np.asarray(target_cloud.compute_point_cloud_distance(source_cloud_transformed))
    chamfer_distance = dist_sim_to_real.mean() + dist_real_to_sim.mean()

    print(f"Viewpoint {i}:")
    print("Distance (Simulation → Real):", dist_sim_to_real.mean())
    print("Distance (Real → Simulation):", dist_real_to_sim.mean())
    print("Difference (Sim - Real):", dist_sim_to_real.mean() - dist_real_to_sim.mean())
    print("Chamfer Distance:", chamfer_distance)